# 00. Program architecture

この notebook は、前処理・因果探索・因果推論を通底するプログラム構造を説明します。

扱うもの:

- file loading と path resolution
- YAML config と typed config object
- `ExecutionPlan` / `StagePlan`
- `PipelineCommandStrategy`: dry-run / validate-only / run
- artifact manifest による再現性管理

因果推論の考え方そのものではなく、「分析を再現可能な pipeline として組むための骨格」を見る notebook です。

## Setup

notebook をどの directory から開いても import できるよう、repository root を探索して `sys.path` を設定します。

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPOSITORY_MARKER = "pyproject.toml"
cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / REPOSITORY_MARKER).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("repository root was not found")

ARTICLE_ROOT = PROJECT_ROOT
SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

for path in (SRC_DIR,):
    value = str(path)
    if value not in sys.path:
        sys.path.insert(0, value)

PROJECT_ROOT

## Layer map

| package | 責務 | ここで混ぜないもの |
|---|---|---|
| `causal_atelier.shared` + `causal_atelier.infrastructure.config` + `causal_atelier.preprocessing.common` | path、YAML、feature semantics、causal design、validation data structure | stage 固有の実行 |
| `causal_atelier.causal.discovery` | discovery 用 feature build、PC/GES/LiNGAM/NOTEARS、discovery artifact | ATE/ATT の推定 |
| `causal_atelier.causal.inference` | edge-weight mode、treatment-effect mode、diagnostics、report | discovery algorithm の実行 |
| `causal_atelier.application` | plan、strategy、executor、cross-stage validation、manifest | stage 内部の統計処理 |

この分割の狙いは、分析上の問いと実行上の責務を混ぜないことです。特に discovery graph の edge と treatment effect は別物です。

In [ ]:
from causal_atelier.infrastructure.config import load_yaml_mapping
from causal_atelier.shared.constants import SUPPORTED_DISCOVERY_ALGORITHMS

config_paths = {
    "pipeline": PROJECT_ROOT / "configs" / "causal" / "inference" / "pipeline.yaml",
    "discovery_analysis": PROJECT_ROOT / "configs" / "causal" / "discovery.yaml",
    "inference_config": PROJECT_ROOT / "configs" / "causal" / "inference" / "defaults.yaml",
}

pd.DataFrame([
    {"name": name, "path": path, "exists": path.exists()}
    for name, path in config_paths.items()
])

## PlanningStrategy 構成

実装名としては `PipelinePlanner` と `PipelineCommandStrategy` です。

- `PipelinePlanner`: CLI args と YAML から `ExecutionPlan` を作る。
- `DryRunStrategy`: stage を実行せず plan を表示する。
- `ValidateOnlyStrategy`: stage を実行せず cross-stage validation だけ行う。
- `RunStrategy`: validation 後に discovery、inference stage を順に実行する。

`PlanningStrategy` という 1 クラスがあるわけではなく、planning と command strategy を分けている点が設計上のポイントです。

In [ ]:
from causal_atelier.interfaces.cli.pipeline import parse_args
from causal_atelier.application.pipeline.planning import PipelinePlanner
from causal_atelier.application.pipeline.strategies import DryRunStrategy, ValidateOnlyStrategy, select_strategy

tutorial_artifacts = PROJECT_ROOT / "artifacts" / "experiments"
args = parse_args([
    "--project-root", str(PROJECT_ROOT),
    "--run-id", "architecture-tutorial",
    "--discovery-algorithms", "pc",
    "--discovery-output-dir", str(tutorial_artifacts / "causal_discovery"),
    "--inference-output-dir", str(tutorial_artifacts / "causal_inference"),
])
plan = PipelinePlanner(PROJECT_ROOT).build_plan(args, strategy_name="dry_run")

pd.DataFrame([
    {
        "stage": stage.name,
        "enabled": stage.enabled,
        "output_dir": stage.output_paths["output_dir"],
        "manifest": stage.output_paths["manifest"],
    }
    for stage in plan.stages
])

In [ ]:
for stage in plan.stages:
    print(f"[{stage.name}] child CLI args")
    print(" ".join(stage.resolved_args))
    print()

## Strategy の差分

`dry-run` は plan を返すだけです。`validate-only` は config と stage 間 contract を確認します。`run` は stage runner を呼び、最後に manifest を書きます。

因果推論上の注意: validation は識別仮定を証明しません。検出できるのは、設定ファイル、feature semantics、causal design、adjustment set の機械的な破綻です。

In [ ]:
dry_result = DryRunStrategy().execute(plan)
validation_result = ValidateOnlyStrategy().execute(plan)

pd.DataFrame([
    {"strategy": dry_result.strategy, "status": dry_result.status, "payload_keys": list(dry_result.payload.keys())},
    {"strategy": validation_result.strategy, "status": validation_result.status, "payload_keys": None},
])

In [ ]:
pd.DataFrame(validation_result.validation.to_dicts()) if validation_result.validation.issues else pd.DataFrame([{"validation": "ok"}])

## Artifact manifest

manifest は、stage output と config hash を追跡するための台帳です。

So what?: 因果分析では、推定値だけを保存しても再現性は足りません。どの feature config、どの causal design、どの discovery artifact を使ったかを追跡する必要があります。